# TP2 — Prévision de la consommation électrique (BiLSTM + Attention)

**Objectif :** prédire la consommation électrique horaire (`Global_active_power`) à partir de
l'historique des 48 dernières heures, sur le dataset **UCI Household Power Consumption**.

**Approche :** réseau **BiLSTM (bidirectionnel, 2 couches)** couplé à un **mécanisme d'attention**
dot-product, entraîné avec une **Huber loss** (robuste aux pics de consommation).

> Notebook conçu pour s'exécuter directement sur **Google Colab**.

## Cellule 1 — Installation des dépendances

Installation des bibliothèques nécessaires (à exécuter sur Colab).

In [ ]:
!pip install torch torchvision numpy pandas matplotlib seaborn scikit-learn requests

## Cellule 2 — Téléchargement automatique du dataset UCI

On télécharge et décompresse l'archive UCI Household Power Consumption directement dans le répertoire courant.

In [ ]:
# Téléchargement et extraction de l'archive ZIP du dataset UCI
import requests, zipfile, io

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall(".")  # extrait household_power_consumption.txt dans le dossier courant
print("Dataset extrait :", z.namelist())

## Cellule 3 — Imports et configuration

Imports des bibliothèques et fixation de la graine aléatoire (`seed=42`).

In [ ]:
# Bibliothèques de base
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Outils sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Fixation de la graine pour la reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

## Cellule 4 — Chargement et nettoyage

- Parsing de `Date` + `Time` en index `datetime`.
- Remplacement des `?` par `NaN` puis **interpolation linéaire**.
- Cible : `Global_active_power`.
- **Rééchantillonnage horaire** (moyenne).

In [ ]:
# Chargement du fichier : séparateur ';', les '?' marquent les valeurs manquantes
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",
    na_values=["?"],          # '?' -> NaN
    low_memory=False,
)

# Construction d'un index datetime à partir des colonnes Date et Time
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")
df = df.set_index("datetime").drop(columns=["Date", "Time"])

# Conversion en numérique (les colonnes sont lues comme texte à cause des '?')
df = df.apply(pd.to_numeric, errors="coerce")

# Interpolation linéaire des valeurs manquantes
df = df.interpolate(method="linear").bfill().ffill()

print(f"Dimensions : {df.shape}")
df.head()

In [ ]:
# Rééchantillonnage horaire : on garde la moyenne de Global_active_power par heure
hourly = df[["Global_active_power"]].resample("1H").mean()
hourly = hourly.interpolate(method="linear")  # comble d'éventuels trous après resample

print(f"Série horaire : {hourly.shape}")
hourly.head()

## Cellule 5 — Feature engineering

On ajoute des variables temporelles cycliques (encodage sin/cos pour l'heure, le jour de la
semaine et le mois) ainsi qu'une **moyenne glissante sur 24h**.

In [ ]:
# Encodage cyclique : sin/cos pour capturer la périodicité sans rupture (23h -> 0h)
idx = hourly.index
hourly["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
hourly["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
hourly["dow_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
hourly["dow_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
hourly["month_sin"] = np.sin(2 * np.pi * (idx.month - 1) / 12)
hourly["month_cos"] = np.cos(2 * np.pi * (idx.month - 1) / 12)

# Moyenne glissante 24h de la consommation (tendance journalière)
hourly["roll_mean_24h"] = hourly["Global_active_power"].rolling(window=24, min_periods=1).mean()

hourly.head()

## Cellule 6 — Normalisation et fenêtrage

- **MinMaxScaler** sur toutes les features (fit sur le train uniquement).
- **Fenêtre glissante** de 48h pour prédire la consommation de l'heure suivante (h+1).
- Split **70% / 15% / 15%** (train / val / test) en respectant l'ordre temporel (pas de shuffle).

In [ ]:
# Ordre des colonnes : la cible (Global_active_power) est en première position (index 0)
feature_cols = ["Global_active_power", "hour_sin", "hour_cos", "dow_sin", "dow_cos",
                "month_sin", "month_cos", "roll_mean_24h"]
data = hourly[feature_cols].values.astype(np.float32)
TARGET_IDX = 0  # indice de la colonne cible

# Découpage temporel des indices (sans shuffle)
n = len(data)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

# MinMaxScaler ajusté UNIQUEMENT sur la partie train (évite la fuite d'information)
scaler = MinMaxScaler()
scaler.fit(data[:train_end])
data_scaled = scaler.transform(data)

In [ ]:
WINDOW = 48  # 48h d'historique en entrée

def make_sequences(series, window):
    """Construit les fenêtres glissantes (X) et la cible h+1 (y) à partir d'une série multivariée."""
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i + window])              # 48h de toutes les features
        y.append(series[i + window, TARGET_IDX])    # consommation à h+1
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Génération des séquences sur l'ensemble de la série
X_all, y_all = make_sequences(data_scaled, WINDOW)

# Découpage train/val/test en respectant l'ordre temporel.
# On décale les bornes de WINDOW car chaque séquence consomme 48h d'historique.
tr_end = train_end - WINDOW
va_end = val_end - WINDOW
X_train, y_train = X_all[:tr_end], y_all[:tr_end]
X_val, y_val = X_all[tr_end:va_end], y_all[tr_end:va_end]
X_test, y_test = X_all[va_end:], y_all[va_end:]

print(f"Train : {X_train.shape} | Val : {X_val.shape} | Test : {X_test.shape}")

# DataLoaders PyTorch
BATCH_SIZE = 64
train_dl = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=BATCH_SIZE, shuffle=False)
test_dl = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=BATCH_SIZE, shuffle=False)
n_features = X_train.shape[2]

## Cellule 7 — Modèle BiLSTM + Attention

- **`Attention`** : attention dot-product calculée sur l'ensemble des sorties temporelles du BiLSTM, produisant un vecteur de contexte (somme pondérée des pas de temps).
- **`BiLSTMAttention`** : LSTM bidirectionnel (2 couches, hidden=64, dropout=0.2), couche d'attention, puis couche linéaire finale vers la prédiction scalaire.

In [ ]:
class Attention(nn.Module):
    """Attention dot-product sur les sorties d'un BiLSTM.

    Un vecteur de score apprenable mesure l'importance de chaque pas de temps ;
    softmax -> poids d'attention ; le contexte est la somme pondérée des sorties.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # Vecteur de scoring (produit scalaire avec chaque sortie temporelle)
        self.score = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out):
        # lstm_out : (batch, seq_len, hidden_dim)
        scores = self.score(lstm_out).squeeze(-1)        # (batch, seq_len)
        weights = torch.softmax(scores, dim=1)           # poids d'attention normalisés
        # Contexte = somme pondérée des sorties sur la dimension temporelle
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)  # (batch, hidden_dim)
        return context, weights

In [ ]:
class BiLSTMAttention(nn.Module):
    """BiLSTM (2 couches) + attention + couche de sortie pour la régression."""
    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,      # lecture avant + arrière
            dropout=dropout,
        )
        # Sortie bidirectionnelle => dimension = 2 * hidden_size
        self.attention = Attention(hidden_size * 2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)              # (batch, seq_len, 2*hidden)
        context, _ = self.attention(lstm_out)   # vecteur de contexte pondéré
        return self.fc(context).squeeze(1)      # prédiction scalaire (batch,)

## Cellule 8 — Entraînement (30 époques)

- Optimiseur **Adam** (`lr=1e-3`).
- Loss **HuberLoss** (robuste aux pics de consommation).
- Scheduler **ReduceLROnPlateau** (`patience=5`, `factor=0.5`) sur la val_loss.
- **Early stopping** (`patience=7`) sur la val_loss, avec restauration des meilleurs poids.
- Affichage de `train_loss` / `val_loss` à chaque époque.

In [ ]:
# Instanciation du modèle, de la loss, de l'optimiseur et du scheduler
model = BiLSTMAttention(n_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.HuberLoss()  # robuste aux valeurs extrêmes (pics de consommation)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

EPOCHS = 30
ES_PATIENCE = 7  # early stopping sur la val_loss

def run_epoch(loader, train_mode):
    """Exécute une époque (train ou val) et renvoie la loss moyenne."""
    model.train() if train_mode else model.eval()
    running, total = 0.0, 0
    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if train_mode:
                optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            if train_mode:
                loss.backward()
                optimizer.step()
            running += loss.item() * xb.size(0)
            total += xb.size(0)
    return running / total

history = {"train_loss": [], "val_loss": []}
best_val = float("inf")
epochs_no_improve = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_dl, train_mode=True)
    val_loss = run_epoch(val_dl, train_mode=False)
    scheduler.step(val_loss)  # ajuste le lr si la val_loss stagne

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}")

    # Early stopping : sauvegarde des meilleurs poids
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= ES_PATIENCE:
            print(f"Early stopping à l'époque {epoch} (patience={ES_PATIENCE}).")
            break

# Restauration des meilleurs poids
if best_state is not None:
    model.load_state_dict(best_state)

## Cellule 9 — Courbes

- Évolution de la loss (train vs validation).
- Prédictions vs valeurs réelles sur 7 jours du test set, **après dénormalisation** (retour en kW).
- Zoom sur les pics de consommation.

In [ ]:
# Courbe de loss train vs validation
plt.figure(figsize=(9, 5))
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], marker="o", label="Train")
plt.plot(range(1, len(history["val_loss"]) + 1), history["val_loss"], marker="o", label="Validation")
plt.title("Huber loss par époque")
plt.xlabel("Époque"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Prédictions du modèle sur le test set
model.eval()
preds_scaled = []
with torch.no_grad():
    for xb, _ in test_dl:
        preds_scaled.append(model(xb.to(device)).cpu().numpy())
preds_scaled = np.concatenate(preds_scaled)

# Dénormalisation : on inverse le MinMaxScaler uniquement sur la colonne cible (index 0).
# MinMaxScaler (plage par défaut [0,1]) : valeur = scaled * data_range_ + data_min_
target_min = scaler.data_min_[TARGET_IDX]
target_range = scaler.data_range_[TARGET_IDX]
y_pred = preds_scaled * target_range + target_min
y_true = y_test * target_range + target_min

In [ ]:
# Affichage sur 7 jours (168 heures) du test set, valeurs dénormalisées
H = 24 * 7  # 168 heures
plt.figure(figsize=(15, 5))
plt.plot(y_true[:H], label="Réel", linewidth=1.5)
plt.plot(y_pred[:H], label="Prédit", linewidth=1.5, alpha=0.8)
plt.title("Consommation réelle vs prédite — 7 jours (test set)")
plt.xlabel("Heure"); plt.ylabel("Global active power (kW)")
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

# Zoom sur les pics : on cible la fenêtre de 48h autour du maximum de consommation
peak = int(np.argmax(y_true[:H]))
lo, hi = max(0, peak - 24), min(H, peak + 24)
plt.figure(figsize=(12, 4))
plt.plot(range(lo, hi), y_true[lo:hi], marker="o", label="Réel")
plt.plot(range(lo, hi), y_pred[lo:hi], marker="o", label="Prédit", alpha=0.8)
plt.title("Zoom sur un pic de consommation")
plt.xlabel("Heure"); plt.ylabel("Global active power (kW)")
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

## Cellule 10 — Métriques finales

Évaluation sur l'ensemble du test set (valeurs dénormalisées) : **MAE**, **RMSE**, **R²** et **MAPE**, présentées dans un tableau pandas.

In [ ]:
# Calcul des métriques sur l'ensemble du test set (valeurs en kW)
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

# MAPE : on évite la division par zéro en masquant les valeurs réelles quasi nulles
mask = np.abs(y_true) > 1e-3
mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# Tableau récapitulatif propre
metrics = pd.DataFrame({
    "Métrique": ["MAE", "RMSE", "R²", "MAPE (%)"],
    "Valeur": [round(mae, 4), round(rmse, 4), round(r2, 4), round(mape, 2)],
})
metrics